# Continue Seq2Seq From Best Checkpoint

Notebook nay warm-start tu folder `best` cua cac seq2seq full-model checkpoint trong old Kaggle output, train tiep voi learning rate nho, chon best theo ROUGE-L, roi evaluate tren full test.

Mac dinh doc old output root theo Kaggle input UI:

```text
/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs
```

LoRA adapter runs bi skip trong notebook nay vi can load adapter de train tiep rieng. Full checkpoint co `best/model.safetensors` se duoc train tiep.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'continue_best_outputs'
REPORT_DIR = OUTPUT_ROOT / '_report'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'
TEST_FILE = DATA_DIR / 'test-00000-of-00001.parquet'

OLD_OUTPUT_ROOTS = [
    Path('/kaggle/input/nlp-vit5-ep3-old-output/summarization_outputs'),
    Path('/kaggle/input/datasets/anhnguyen0812/nlp-vit5-ep3-old-output/summarization_outputs'),
]

RUN_GLOB = 'vit5_base_ep3_t4x2'  # best full-model run; lora adapter run khong train tiep trong notebook nay
CONTINUE_MAX_STEPS = 700
TRAIN_MAX_SAMPLES = None  # None = full train
EVAL_MAX_SAMPLES = 500
TEST_MAX_SAMPLES = None  # None = full test
TEST_EVAL_BATCH_SIZE = 4
TEST_NUM_BEAMS = 4
TEST_MAX_NEW_TOKENS = 160
OVERWRITE_RUNS = False
OVERWRITE_TEST = True

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


In [ ]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)

for path in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    print(path, path.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError('Attach dataset anhnguyen0812/nlp-vietnamese-sumarization first.')

import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('NUM_GPUS:', NUM_GPUS)


## Discover Best Checkpoints

Notebook chi train tiep full seq2seq model folders co `best/model.safetensors`. LoRA folder co `best/adapter_model.safetensors` se bi skip.


In [ ]:
def has_full_model(best_dir):
    return (best_dir / 'model.safetensors').exists() or (best_dir / 'pytorch_model.bin').exists()

def has_lora_adapter(best_dir):
    return (best_dir / 'adapter_model.safetensors').exists() or (best_dir / 'adapter_config.json').exists()

def discover_runs():
    runs = []
    for root in OLD_OUTPUT_ROOTS:
        print('OLD_OUTPUT_ROOT:', root, root.exists())
        if not root.exists():
            continue
        for run_dir in sorted(root.glob(RUN_GLOB)):
            if not run_dir.is_dir():
                continue
            best = run_dir / 'best'
            resolved = run_dir / 'resolved_config.json'
            if not best.exists() or not resolved.exists():
                print('SKIP missing best/resolved_config:', run_dir)
                continue
            if has_lora_adapter(best) and not has_full_model(best):
                print('SKIP lora adapter run:', run_dir.name)
                continue
            if not has_full_model(best):
                print('SKIP no full model artifact:', run_dir.name)
                continue
            runs.append(run_dir)
    unique = []
    seen = set()
    for run_dir in runs:
        key = str(run_dir.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(run_dir)
    return unique

SOURCE_RUNS = discover_runs()
print('SOURCE_RUNS:', [p.name for p in SOURCE_RUNS])
if not SOURCE_RUNS:
    raise FileNotFoundError('Khong tim thay full model best checkpoint. Kiem tra OLD_OUTPUT_ROOTS/RUN_GLOB hoac attach dataset output dung.')


## Train Continue Phase

Day la warm-start tu `best`, khong resume optimizer state. Ly do: `best` la model export on dinh nhat, con checkpoint optimizer trong Kaggle input la read-only va co the dung state old phase khong hop voi output moi.


In [ ]:
import yaml

GENERATED_CONFIG_DIR = repo / 'configs' / '_generated_continue_best'
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def load_json(path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def save_yaml(data, path):
    with path.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

def model_family(run_dir, cfg):
    name = str(cfg.get('model', {}).get('name_or_path', '')).lower() + ' ' + run_dir.name.lower()
    if 'bartpho' in name:
        return 'bartpho'
    if 'vit5' in name or 't5' in name:
        return 'vit5'
    return 'seq2seq'

def continue_overrides(family):
    if family == 'bartpho':
        return {
            'learning_rate': 1e-5,
            'generation': {
                'max_length': 192,
                'min_length': 70,
                'num_beams': 4,
                'length_penalty': 1.0,
                'no_repeat_ngram_size': 3,
                'repetition_penalty': 1.03,
                'early_stopping': True,
            },
        }
    return {
        'learning_rate': 1e-5,
        'generation': {
            'max_length': 192,
            'min_length': 70,
            'num_beams': 6,
            'length_penalty': 1.05,
            'no_repeat_ngram_size': 3,
            'repetition_penalty': 1.05,
            'early_stopping': True,
        },
    }

def make_continue_config(run_dir):
    old_cfg = load_json(run_dir / 'resolved_config.json')
    cfg = json.loads(json.dumps(old_cfg))
    family = model_family(run_dir, cfg)
    overrides = continue_overrides(family)
    out_name = f'{run_dir.name}_continue_best_{CONTINUE_MAX_STEPS}s'
    out_dir = OUTPUT_ROOT / out_name

    cfg.setdefault('model', {})['name_or_path'] = str(run_dir / 'best')
    cfg['model']['cache_dir'] = None
    cfg.setdefault('data', {}).update({
        'train_file': str(TRAIN_FILE),
        'valid_file': str(VALID_FILE),
        'max_source_length': 1024,
        'max_target_length': 192,
        'preprocessing_num_proc': 2,
        'max_train_samples': TRAIN_MAX_SAMPLES,
        'max_eval_samples': EVAL_MAX_SAMPLES,
    })
    cfg.setdefault('training', {}).update({
        'output_dir': str(out_dir),
        'overwrite_output_dir': OVERWRITE_RUNS,
        'precision': 'fp16',
        'per_device_train_batch_size': 2,
        'per_device_eval_batch_size': 8,
        'gradient_accumulation_steps': 4,
        'num_train_epochs': 1,
        'max_steps': CONTINUE_MAX_STEPS,
        'learning_rate': overrides['learning_rate'],
        'weight_decay': 0.01,
        'warmup_ratio': 0.03,
        'lr_scheduler_type': 'cosine',
        'label_smoothing_factor': 0.05,
        'dropout': 0.1,
        'optim': 'adamw_torch',
        'gradient_checkpointing': True,
        'freeze_encoder': False,
        'strategy': 'steps',
        'save_strategy': 'steps',
        'eval_steps': 250,
        'save_steps': 250,
        'logging_steps': 50,
        'save_total_limit': 2,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'rougeL',
        'greater_is_better': True,
        'early_stopping_patience': 3,
        'ddp_find_unused_parameters': False,
        'save_safetensors': True,
    })
    cfg['training'].pop('resume_from_checkpoint', None)
    cfg['generation'] = overrides['generation']
    cfg.setdefault('lora', {})['enabled'] = False

    config_path = GENERATED_CONFIG_DIR / f'{out_name}.yaml'
    save_yaml(cfg, config_path)
    return {
        'source_run': run_dir.name,
        'family': family,
        'name': out_name,
        'source_best': run_dir / 'best',
        'config_path': config_path,
        'output_dir': out_dir,
    }

CONTINUE_RUNS = [make_continue_config(run_dir) for run_dir in SOURCE_RUNS]
for item in CONTINUE_RUNS:
    print(json.dumps({k: str(v) for k, v in item.items()}, ensure_ascii=False, indent=2))
    print(Path(item['config_path']).read_text(encoding='utf-8')[:2000])


In [ ]:
def launch_train(config_path):
    rel_config = Path(config_path).relative_to(repo).as_posix()
    train_cmd = f'-m vn_summarization.train --config {rel_config}'
    if NUM_GPUS >= 2:
        cmd = f'{sys.executable} -m accelerate.commands.launch --multi_gpu --num_processes {NUM_GPUS} --num_machines 1 --mixed_precision fp16 --dynamo_backend no {train_cmd}'
    else:
        cmd = f'{sys.executable} -u {train_cmd}'
    return run(cmd, cwd=repo)

train_start = time.time()
for item in CONTINUE_RUNS:
    out_dir = Path(item['output_dir'])
    if (out_dir / 'best' / 'model.safetensors').exists() and not OVERWRITE_RUNS:
        print('SKIP existing trained run:', out_dir)
        continue
    if out_dir.exists() and OVERWRITE_RUNS:
        shutil.rmtree(out_dir)
    print('\n' + '=' * 100)
    print('TRAIN CONTINUE:', item['name'])
    t0 = time.time()
    launch_train(item['config_path'])
    print('DONE:', item['name'], 'elapsed_hours=', round((time.time() - t0) / 3600, 3))
print('TRAIN_ALL elapsed_hours=', round((time.time() - train_start) / 3600, 3))


In [ ]:
TEST_OUT_DIR = OUTPUT_ROOT / '_test_eval'
if TEST_OUT_DIR.exists() and OVERWRITE_TEST:
    shutil.rmtree(TEST_OUT_DIR)
TEST_OUT_DIR.mkdir(parents=True, exist_ok=True)

args = (
    f'--runs_root {OUTPUT_ROOT} '
    f'--run_glob "*_continue_best_*" '
    f'--test_file {TEST_FILE} '
    f'--out_dir {TEST_OUT_DIR} '
    f'--eval_batch_size {TEST_EVAL_BATCH_SIZE} '
    f'--generation_num_beams {TEST_NUM_BEAMS} '
    f'--generation_max_new_tokens {TEST_MAX_NEW_TOKENS}'
)
if TEST_MAX_SAMPLES is not None:
    args += f' --max_test_samples {TEST_MAX_SAMPLES}'

run(f'{sys.executable} -u -m vn_summarization.evaluate_runs_on_test {args}', cwd=repo)


In [ ]:
def read_json(path):
    if not Path(path).exists():
        return {}
    with Path(path).open('r', encoding='utf-8') as f:
        return json.load(f)

rows = []
result_csv = TEST_OUT_DIR / 'test_results.csv'
if result_csv.exists():
    with result_csv.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))

summary_rows = []
for item in CONTINUE_RUNS:
    out_dir = Path(item['output_dir'])
    train_metrics = read_json(out_dir / 'train_results.json')
    eval_metrics = read_json(out_dir / 'eval_results.json')
    test_row = next((row for row in rows if row.get('run') == item['name']), {})
    summary_rows.append({
        'source_run': item['source_run'],
        'continued_run': item['name'],
        'family': item['family'],
        'max_steps': CONTINUE_MAX_STEPS,
        'eval_rouge1': eval_metrics.get('eval_rouge1', ''),
        'eval_rouge2': eval_metrics.get('eval_rouge2', ''),
        'eval_rougeL': eval_metrics.get('eval_rougeL', ''),
        'eval_loss': eval_metrics.get('eval_loss', ''),
        'test_rouge1': test_row.get('rouge1', ''),
        'test_rouge2': test_row.get('rouge2', ''),
        'test_rougeL': test_row.get('rougeL', ''),
        'test_gen_len': test_row.get('gen_len', ''),
        'train_runtime': train_metrics.get('train_runtime', ''),
        'source_best': str(item['source_best']),
    })

columns = ['source_run', 'continued_run', 'family', 'max_steps', 'eval_rouge1', 'eval_rouge2', 'eval_rougeL', 'eval_loss', 'test_rouge1', 'test_rouge2', 'test_rougeL', 'test_gen_len', 'train_runtime', 'source_best']
csv_path = REPORT_DIR / 'continue_best_summary.csv'
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(summary_rows)

lines = ['# Continue Best Summary', '', '| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
for row in summary_rows:
    lines.append('| ' + ' | '.join(str(row.get(col, '')) for col in columns) + ' |')
lines.append('')
md_path = REPORT_DIR / 'continue_best_summary.md'
md_path.write_text('\n'.join(lines), encoding='utf-8')
print(md_path.read_text(encoding='utf-8'))
print('CSV:', csv_path)
print('TEST_RESULTS:', TEST_OUT_DIR / 'test_results.md')


In [ ]:
zip_path = WORKING / 'continue_best_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt', '.model', '.safetensors'}

def keep_file(path):
    if not path.is_file() or path.suffix not in keep_suffixes:
        return False
    if any(part.startswith('checkpoint-') for part in path.parts):
        return False
    return True

files = [p for p in OUTPUT_ROOT.rglob('*') if keep_file(p)]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP:', zip_path)
for file in sorted(files):
    print(file)
